# EXACT 2026 — Root Data Audit: RAW vs CLEAN/V5

Notebook này audit **data gốc** và **data đang dùng v5** theo tam giác:

```text
label ↔ explanation ↔ premises/FOL/idx
```

Nguyên tắc: **không dùng Z3 như quan tòa tuyệt đối**. Notebook chỉ tìm các case mâu thuẫn/đáng review, in trực tiếp bảng chính trong output cell, và lưu CSV/JSON vào `/kaggle/working/audit_raw_vs_v5`.


In [1]:
# ==================================================================
# STAGE 0 -- Imports & path resolver
# ==================================================================
import os, re, json, csv, math, hashlib, textwrap, statistics
from pathlib import Path
from collections import Counter, defaultdict
from difflib import SequenceMatcher

try:
    import pandas as pd
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas"], check=False)
    import pandas as pd

pd.set_option("display.max_colwidth", 220)
pd.set_option("display.max_rows", 200)

print("="*80)
print("ROOT AUDIT NOTEBOOK -- imports OK")
print("="*80)


def _resolve(cands, label="path"):
    import glob
    expanded = []
    for p in cands:
        hits = sorted(glob.glob(p, recursive=True)) if any(ch in p for ch in "*?[") else [p]
        expanded.extend(hits)
    for p in expanded:
        if os.path.exists(p):
            print(f"  resolved {label}: {p}")
            return p
    print(f"  WARNING {label}: none found; using first candidate: {cands[0]}")
    return cands[0]

# Chồng đã cung cấp 2 path Kaggle chính xác. Các path sau chỉ là fallback.
RAW_PATH = _resolve([
    "/kaggle/input/datasets/nguyenminhtric/test-pipeline/Logic_Based_Educational_Queries.json",
    "/kaggle/input/test-pipeline/Logic_Based_Educational_Queries.json",
    "/kaggle/input/**/Logic_Based_Educational_Queries.json",
    "/mnt/data/Logic_Based_Educational_Queries(2).json",
    "/mnt/data/Logic_Based_Educational_Queries.json",
], "RAW411")

CLEAN_PATH = _resolve([
    "/kaggle/input/datasets/nguyenminhtric/test-pipeline/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json",
    "/kaggle/input/test-pipeline/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json",
    "/kaggle/input/**/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json",
    "/mnt/data/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy(1).json",
    "/mnt/data/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json",
], "CLEAN/V5")

OUT_DIR = Path("/kaggle/working/audit_raw_vs_v5") if Path("/kaggle/working").exists() else Path("/mnt/data/audit_raw_vs_v5")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("RAW_PATH  =", RAW_PATH)
print("CLEAN_PATH=", CLEAN_PATH)
print("OUT_DIR   =", OUT_DIR)


ROOT AUDIT NOTEBOOK -- imports OK
  resolved RAW411: /kaggle/input/datasets/nguyenminhtric/test-pipeline/Logic_Based_Educational_Queries.json
  resolved CLEAN/V5: /kaggle/input/datasets/nguyenminhtric/test-pipeline/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json
RAW_PATH  = /kaggle/input/datasets/nguyenminhtric/test-pipeline/Logic_Based_Educational_Queries.json
CLEAN_PATH= /kaggle/input/datasets/nguyenminhtric/test-pipeline/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json
OUT_DIR   = /kaggle/working/audit_raw_vs_v5


In [2]:
# ==================================================================
# STAGE 1 -- Load data and flatten question-level rows
# ==================================================================
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

raw = load_json(RAW_PATH)
clean = load_json(CLEAN_PATH)

print(f"RAW   records: {len(raw)}")
print(f"CLEAN records: {len(clean)}")

LABELS = ["A", "B", "C", "D", "YES", "NO", "UNKNOWN"]
LABEL_ORDER = {x:i for i,x in enumerate(LABELS)}


def norm_ws(s):
    return re.sub(r"\s+", " ", str(s).strip())


def norm_text(s):
    s = norm_ws(s).lower()
    s = re.sub(r"[“”\"'`]", "", s)
    s = re.sub(r"\s+", " ", s)
    return s


def norm_answer(a):
    x = str(a).strip()
    xu = x.upper()
    if xu in ("YES", "TRUE"):
        return "YES"
    if xu in ("NO", "FALSE"):
        return "NO"
    if xu in ("UNKNOWN", "UNCERTAIN", "CANNOT BE DETERMINED", "UNDETERMINED"):
        return "UNKNOWN"
    if xu in ("A", "B", "C", "D"):
        return xu
    return xu


def h(s, n=16):
    return hashlib.md5(str(s).encode("utf-8")).hexdigest()[:n]


def record_key(rec):
    prem = "||".join(norm_text(x) for x in rec.get("premises-NL", []))
    qs = "||".join(norm_text(x) for x in rec.get("questions", []))
    return h(prem + "###" + qs, 24)


def q_key(rec, qi):
    prem = "||".join(norm_text(x) for x in rec.get("premises-NL", []))
    q = norm_text((rec.get("questions") or [""])[qi])
    return h(prem + "###" + q, 24)


def flatten(data, name):
    rows=[]
    for ri, rec in enumerate(data):
        questions = rec.get("questions", []) or []
        answers = rec.get("answers", []) or []
        exps = rec.get("explanation", []) or []
        idxs = rec.get("idx", []) or []
        for qi, q in enumerate(questions):
            ans = answers[qi] if qi < len(answers) else None
            exp = exps[qi] if qi < len(exps) else ""
            idx = idxs[qi] if qi < len(idxs) else None
            rows.append({
                "dataset": name,
                "record_i": ri,
                "q_idx": qi,
                "record_key": record_key(rec),
                "q_key": q_key(rec, qi),
                "answer": ans,
                "answer_norm": norm_answer(ans),
                "question": q,
                "explanation": exp,
                "idx": idx,
                "n_premises": len(rec.get("premises-NL", []) or []),
                "n_fol": len(rec.get("premises-FOL", []) or []),
            })
    return pd.DataFrame(rows)

raw_q = flatten(raw, "raw")
clean_q = flatten(clean, "clean")

print(f"RAW   questions: {len(raw_q)}")
print(f"CLEAN questions: {len(clean_q)}")
print("Sample raw rows:")
display(raw_q.head(3))


RAW   records: 411
CLEAN records: 399
RAW   questions: 808
CLEAN questions: 784
Sample raw rows:


,dataset,record_i,q_idx,record_key,q_key,answer,answer_norm,question,explanation,idx,n_premises,n_fol
0,raw,0,0,5d3c2a6f961e48f76c1fceb3,d4a952cfe9f8ba9d9db3bad5,A,A,"Which conclusion follows with the fewest premises?\nA. If a Python project is not optimized, then it is not well-tested\nB. If all Python projects are optimized, then all Python projects are well-structured\nC. If a ...","Premise 1 states that if a Python project is well-tested, it is optimized. By logical contraposition, if a project is not optimized, it is not well-tested, supporting option A with the fewest premises. Option B is fa...",[1],14,14
1,raw,0,1,5d3c2a6f961e48f76c1fceb3,bff1bd2d7a1b5409addea440,Yes,YES,"Does it follow that if all Python projects are well-structured, then all Python projects are optimized, according to the premises?","Premise 10 confirms all Python projects are well-structured. Premise 7 states that well-structured projects are optimized, implying all projects are optimized, so the statement that well-structured projects imply opt...","[7, 10]",14,14
2,raw,1,0,2f4bb6cfefec51ff60a9fa08,dceb1e7479b6d49b6c239b22,C,C,"Based on the above premises, which is the strongest conclusion?\nA. Sophia qualifies for the university scholarship\nB. Sophia needs a faculty recommendation to qualify for the scholarship\nC. Sophia is eligible for ...","Premises 7 and 8 confirm Sophia completed the core curriculum and passed the science assessment, satisfying premise 1 for advanced courses. Premise 9’s research methodology and premise 2 make her eligible for the int...","[1, 2, 4, 5, 7, 8, 9, 10, 11]",11,11


In [3]:
# ==================================================================
# STAGE 2 -- Distribution comparison
# ==================================================================
def dist_table(raw_q, clean_q):
    labels = sorted(
        set(raw_q.answer_norm.dropna()) | set(clean_q.answer_norm.dropna()),
        key=lambda x: LABEL_ORDER.get(x, 999)
    )
    rows=[]
    for lab in labels:
        r = int((raw_q.answer_norm == lab).sum())
        c = int((clean_q.answer_norm == lab).sum())
        rows.append({
            "label": lab,
            "raw_count": r,
            "clean_count": c,
            "delta_clean_minus_raw": c-r,
            "raw_pct": round(100*r/max(len(raw_q),1), 2),
            "clean_pct": round(100*c/max(len(clean_q),1), 2),
        })
    return pd.DataFrame(rows)

dist = dist_table(raw_q, clean_q)
print("="*80)
print("LABEL DISTRIBUTION SHIFT")
print("="*80)
display(dist)

print("Question count delta:", len(clean_q) - len(raw_q))
print("Record count delta  :", len(clean) - len(raw))

summary = {
    "raw_records": len(raw),
    "clean_records": len(clean),
    "raw_questions": len(raw_q),
    "clean_questions": len(clean_q),
    "question_delta": len(clean_q)-len(raw_q),
    "record_delta": len(clean)-len(raw),
}

dist.to_csv(OUT_DIR / "audit_label_distribution_shift.csv", index=False)
print("Saved:", OUT_DIR / "audit_label_distribution_shift.csv")


LABEL DISTRIBUTION SHIFT


,label,raw_count,clean_count,delta_clean_minus_raw,raw_pct,clean_pct
0,A,105,111,6,13.00,14.16
1,B,39,47,8,4.83,5.99
2,C,20,20,0,2.48,2.55
3,D,19,16,-3,2.35,2.04
4,YES,116,170,54,14.36,21.68
5,NO,300,233,-67,37.13,29.72
6,UNKNOWN,209,187,-22,25.87,23.85


Question count delta: -24
Record count delta  : -12
Saved: /kaggle/working/audit_raw_vs_v5/audit_label_distribution_shift.csv


In [4]:
# ==================================================================
# STAGE 3 -- Match raw vs clean by premise+question key and inspect answer changes
# ==================================================================
raw_small = raw_q[["q_key","record_i","q_idx","answer_norm","answer","question","explanation","idx","record_key"]].rename(columns={
    "record_i":"raw_record_i", "q_idx":"raw_q_idx", "answer_norm":"raw_answer_norm", "answer":"raw_answer",
    "question":"raw_question", "explanation":"raw_explanation", "idx":"raw_idx", "record_key":"raw_record_key"
})
clean_small = clean_q[["q_key","record_i","q_idx","answer_norm","answer","question","explanation","idx","record_key"]].rename(columns={
    "record_i":"clean_record_i", "q_idx":"clean_q_idx", "answer_norm":"clean_answer_norm", "answer":"clean_answer",
    "question":"clean_question", "explanation":"clean_explanation", "idx":"clean_idx", "record_key":"clean_record_key"
})

matched = raw_small.merge(clean_small, on="q_key", how="outer", indicator=True)
matched["answer_changed"] = (matched["raw_answer_norm"].fillna("<MISSING>") != matched["clean_answer_norm"].fillna("<MISSING>")) & (matched["_merge"] == "both")

print("="*80)
print("RAW ↔ CLEAN QUESTION MATCHING")
print("="*80)
print("matched both       :", int((matched._merge == "both").sum()))
print("raw only removed   :", int((matched._merge == "left_only").sum()))
print("clean only added   :", int((matched._merge == "right_only").sum()))
print("answer changed     :", int(matched.answer_changed.sum()))

change_pairs = matched.loc[matched.answer_changed].groupby(["raw_answer_norm","clean_answer_norm"]).size().reset_index(name="count").sort_values("count", ascending=False)
print("Raw → Clean answer changes:")
display(change_pairs)

changes = matched.loc[matched.answer_changed, [
    "raw_record_i","raw_q_idx","clean_record_i","clean_q_idx","raw_answer_norm","clean_answer_norm",
    "raw_question","raw_explanation","clean_explanation","raw_idx","clean_idx"
]].copy()

print("Top 20 changed-answer examples:")
display(changes.head(20))

matched.to_csv(OUT_DIR / "audit_raw_clean_question_matching.csv", index=False)
changes.to_csv(OUT_DIR / "audit_label_changes_raw_vs_clean.csv", index=False)
change_pairs.to_csv(OUT_DIR / "audit_label_change_pairs.csv", index=False)
print("Saved:", OUT_DIR / "audit_label_changes_raw_vs_clean.csv")


RAW ↔ CLEAN QUESTION MATCHING
matched both       : 802
raw only removed   : 18
clean only added   : 0
answer changed     : 83
Raw → Clean answer changes:


,raw_answer_norm,clean_answer_norm,count
5,NO,YES,58
7,UNKNOWN,B,7
6,UNKNOWN,A,6
2,C,A,3
4,D,B,3
1,B,C,1
0,A,C,1
3,D,A,1
8,UNKNOWN,C,1
9,UNKNOWN,D,1


Top 20 changed-answer examples:


,raw_record_i,raw_q_idx,clean_record_i,clean_q_idx,raw_answer_norm,clean_answer_norm,raw_question,raw_explanation,clean_explanation,raw_idx,clean_idx
17,13,1,12.0,1.0,NO,YES,"Can Professor John supervise graduate-level research based on his PhD qualification, according to the premises?","Premise 5 confirms Professor John’s PhD, premise 1 establishes that a PhD qualifies him to teach graduate courses, and premise 4 confirms that teaching qualification enables him to supervise graduate-level research, ...","Premise 5 confirms Professor John’s PhD, premise 1 establishes that a PhD qualifies him to teach graduate courses, and premise 4 confirms that teaching qualification enables him to supervise graduate-level research, ...","[1, 4, 5]","[1, 4, 5]"
23,306,1,299.0,1.0,NO,YES,"Based on the above premises, is the statement true?\nStatement: (PracticeExercises(x) → PerformWell(x))","The statement is true because it is explicitly stated that if a student practices exercises, they will perform well in exams.","The statement is true because it is explicitly stated that if a student practices exercises, they will perform well in exams.",[7],[7]
24,382,1,372.0,1.0,NO,YES,"If Phong has met the foreign language standard for year three, does it follow that Phong is eligible to apply for study abroad?","From premise 3 (Phong has accumulated 70 credits and has met the foreign language standard for year three), we know MeetsLanguageStandard(Phong, YearThree). From premise 14, third-year students can apply for study ab...","From premise 3 (Phong has accumulated 70 credits and has met the foreign language standard for year three), we know MeetsLanguageStandard(Phong, YearThree). From premise 14, third-year students can apply for study ab...",[3],[3]
34,250,1,248.0,1.0,NO,YES,"Is the following statement true? Statement: If a student has not maintained a good academic standing, then they will not be shortlisted for an interview","If a student has not maintained good standing, they will not be shortlisted (premise 2), and at least one student has good standing (premise 3). Thus, the statement is true, requiring steps to confirm consistency.","If a student has not maintained good standing, they will not be shortlisted (premise 2), and at least one student has good standing (premise 3). Thus, the statement is true, requiring steps to confirm consistency.","[2, 3]","[2, 3]"
39,265,1,263.0,1.0,NO,YES,"Based on the above premises, is the statement true?\nStatement: ((QualifiesForScholarship(x) → AttendsTraining(x)) → ForAll(x, StudiesRegularly(x)))","The statement is true because all students study regularly, making the consequent of the implication true, so the entire implication holds.","The statement is true because all students study regularly, making the consequent of the implication true, so the entire implication holds.",[2],[2]
41,243,1,241.0,1.0,NO,YES,Is the following statement true? Statement: All students follow the classroom rules,"All students follow the classroom rules (premise 1), and there exists a student in the student council who follows the rules (premise 2). Thus, the statement is true, requiring steps to confirm consistency.","All students follow the classroom rules (premise 1), and there exists a student in the student council who follows the rules (premise 2). Thus, the statement is true, requiring steps to confirm consistency.","[1, 2]","[1, 2]"
44,207,1,205.0,1.0,NO,YES,"Based on the above premises, is the following statement true?\nStatement: If a student publishes papers, then they conduct research.","The statement is true because it is explicitly given as a premise that if a student publishes papers, then they conduct research (premise 11).","The statement is true because it is explicitly given as a premise that if a student publishes papers, then they conduct research (premise 11).",[11],[11]
54,145,0,144.0,0.0,UNKNOWN,B,"Based on the premises, which of the following can be inferred?\nA) A variable declared with 'let' is im

Saved: /kaggle/working/audit_raw_vs_v5/audit_label_changes_raw_vs_clean.csv


In [5]:
# ==================================================================
# STAGE 4 -- Explanation-vs-label heuristic flags
# These are NOT automatic fixes. They are review candidates.
# ==================================================================
def infer_mc_from_explanation(exp):
    e = norm_ws(exp)
    patterns = [
        r"support(?:ing|s)?\s+option\s+([ABCD])",
        r"option\s+([ABCD])\s+(?:is|as)\s+(?:the\s+)?(?:correct|valid|logically valid|strongest|best)",
        r"making\s+([ABCD])\s+(?:the\s+)?(?:correct|valid|strongest|best)",
        r"hence,?\s+option\s+([ABCD])",
        r"therefore,?\s+option\s+([ABCD])",
        r"answer\s+is\s+([ABCD])\b",
    ]
    for pat in patterns:
        m = re.search(pat, e, flags=re.I)
        if m:
            return m.group(1).upper(), "high", f"pattern:{pat}"
    return None, None, None

POS_PHRASES = [
    "does meet all requirements", "meets all requirements", "satisfying all conditions",
    "therefore, yes", "so yes", "the statement is true", "is true", "entailed", "logically follows",
    "valid and consistent", "supporting the statement", "supports the statement"
]
NEG_PHRASES = [
    "does not meet all requirements", "doesn’t meet all requirements", "doesn't meet all requirements",
    "not meet all requirements", "so no", "therefore, no", "the statement is false", "is false",
    "cannot", "does not follow", "not enough", "isn’t confirmed", "isn't confirmed", "not confirmed",
]
UNK_PHRASES = [
    "cannot be determined", "uncertain", "unknown", "not enough information", "insufficient information",
    "not specified", "not given", "no premise confirms", "is not confirmed"
]


def infer_yn_from_explanation(exp):
    e = norm_text(exp)
    scores = {"YES":0, "NO":0, "UNKNOWN":0}
    for ph in POS_PHRASES:
        if ph in e:
            scores["YES"] += 1
    for ph in NEG_PHRASES:
        if ph in e:
            scores["NO"] += 1
    for ph in UNK_PHRASES:
        if ph in e:
            scores["UNKNOWN"] += 2
    best = max(scores, key=scores.get)
    if scores[best] == 0:
        return None, None, None
    # If unknown signal exists, prefer Unknown over generic negative wording.
    if scores["UNKNOWN"] > 0 and scores["UNKNOWN"] >= scores["NO"]:
        return "UNKNOWN", "medium", "phrase_unknown"
    return best, "low", f"phrase_scores:{scores}"


def flag_exp_label(df, name):
    rows=[]
    for _, r in df.iterrows():
        lab = r.answer_norm
        exp = r.explanation
        pred = conf = why = None
        if lab in ("A","B","C","D"):
            pred, conf, why = infer_mc_from_explanation(exp)
        elif lab in ("YES","NO","UNKNOWN"):
            pred, conf, why = infer_yn_from_explanation(exp)
        disagree = pred is not None and pred != lab
        rows.append({
            "dataset": name,
            "record_i": int(r.record_i),
            "q_idx": int(r.q_idx),
            "label": lab,
            "exp_implied": pred,
            "confidence": conf,
            "why": why,
            "flag_disagree": bool(disagree),
            "question": r.question,
            "explanation": r.explanation,
            "idx": r.idx,
        })
    return pd.DataFrame(rows)

raw_flags = flag_exp_label(raw_q, "raw")
clean_flags = flag_exp_label(clean_q, "clean")

print("="*80)
print("EXPLANATION ↔ LABEL HEURISTIC FLAGS")
print("="*80)
print("Raw flags   :", int(raw_flags.flag_disagree.sum()), "/", len(raw_flags))
print("Clean flags :", int(clean_flags.flag_disagree.sum()), "/", len(clean_flags))

print("Raw high/medium disagreement examples:")
display(raw_flags.query("flag_disagree == True and confidence in ['high','medium']").head(30))
print("Clean high/medium disagreement examples:")
display(clean_flags.query("flag_disagree == True and confidence in ['high','medium']").head(30))

raw_flags.to_csv(OUT_DIR / "audit_raw_explanation_label_flags.csv", index=False)
clean_flags.to_csv(OUT_DIR / "audit_clean_explanation_label_flags.csv", index=False)
print("Saved:", OUT_DIR / "audit_raw_explanation_label_flags.csv")
print("Saved:", OUT_DIR / "audit_clean_explanation_label_flags.csv")


EXPLANATION ↔ LABEL HEURISTIC FLAGS
Raw flags   : 138 / 808
Clean flags : 83 / 784
Raw high/medium disagreement examples:


,dataset,record_i,q_idx,label,exp_implied,confidence,why,flag_disagree,question,explanation,idx
2,raw,1,0,C,A,high,pattern:making\s+([ABCD])\s+(?:the\s+)?(?:correct|valid|strongest|best),True,"Based on the above premises, which is the strongest conclusion?\nA. Sophia qualifies for the university scholarship\nB. Sophia needs a faculty recommendation to qualify for the scholarship\nC. Sophia is eligible for ...","Premises 7 and 8 confirm Sophia completed the core curriculum and passed the science assessment, satisfying premise 1 for advanced courses. Premise 9’s research methodology and premise 2 make her eligible for the int...","[1, 2, 4, 5, 7, 8, 9, 10, 11]"
4,raw,2,0,C,A,high,pattern:support(?:ing|s)?\s+option\s+([ABCD]),True,"Based on the above premises, which conclusion is correct?\nA. Sophia qualifies for the university scholarship\nB. Sophia needs a faculty recommendation to qualify for the scholarship\nC. Sophia is eligible for the in...","Premises 7 and 8 confirm Sophia completed the core curriculum and passed the science assessment, satisfying premise 1 for advanced courses. Premise 9’s research methodology and premise 2 make her eligible for the int...","[1, 2, 4, 5, 7, 8, 9, 10, 11]"
22,raw,11,0,C,A,high,pattern:support(?:ing|s)?\s+option\s+([ABCD]),True,"Based on Alex's status, which statement is correct?\nA. Alex can use equipment but cannot book training without a trainer\nB. Alex can book personal training sessions if assigned a trainer\nC. Alex needs a longer mem...","Premises 3 and 7 confirm Alex paid fees for a valid membership, premise 1 confirms safety orientation, and premise 4 allows equipment use, so Alex can use equipment. Premise 5 requires a trainer to book training, whi...","[1, 3, 4, 7]"
23,raw,11,1,NO,UNKNOWN,medium,phrase_unknown,True,"Does Alex meet all requirements for booking training sessions, according to the premises?","Booking training per premise 5 requires equipment use and having a trainer. Premises 1, 3, 4, and 7 confirm Alex can use equipment, and premises 2 and 6 confirm trainer eligibility, but no premise confirms Alex has a...","[1, 3, 4, 5, 7]"
24,raw,12,0,D,B,high,pattern:support(?:ing|s)?\s+option\s+([ABCD]),True,"Based on the requirements, which statement about Sarah is correct?\nA. She can take advanced classes because she has approval\nB. She cannot take advanced classes due to insufficient completed courses\nC. She is elig...","Premise 3 confirms Sarah has active status, but premise 4 states she completed only 4 courses, which is below the 5-course threshold for eligibility in premise 1. Thus, Sarah cannot take advanced classes due to insuf...","[1, 3, 4]"
603,raw,308,1,NO,UNKNOWN,medium,phrase_unknown,True,"Based on the above premises, is the statement true?\nStatement: Exists(x, ParticipatesInStudyGroup(x))","The statement is not necessarily true because, while not every student participates in a study group, no premise confirms that at least one student does participate.",[3]
625,raw,319,1,NO,UNKNOWN,medium,phrase_unknown,True,Did Dio pass by reading books?,"Premise 5 states Dio borrowed a book and passed, premise 1 confirms he got a book, but premise 2 requires hard work to read, which is uncertain, so passing via reading isn’t guaranteed.","[1, 2, 5]"
698,raw,356,0,A,C,high,pattern:making\s+([ABCD])\s+(?:the\s+)?(?:correct|valid|strongest|best),True,"Which statement can be inferred?\nA. It is both true and false that if there exists at least one student who has completed the Thesis Writing course, then all students are enrolled in Research Methods.\nB. It is fals...","Premise 6 states all students completed Thesis Writing, implying at least one did, and premise 2 confirms all students are enrolled in Research Methods, making C correct.","[2, 6]"
700,raw,357,0,A,B,high,pattern:making\s+([ABCD])\s+(?:the\s+)?(?:correct|valid|strongest|best),True,"Which statement can be inferred?\nA. If a student doesn't complete the Thesis Writing course, then all students are enrolled in 

Clean high/medium disagreement examples:


,dataset,record_i,q_idx,label,exp_implied,confidence,why,flag_disagree,question,explanation,idx
21,clean,10,1,NO,UNKNOWN,medium,phrase_unknown,True,"Does Alex meet all requirements for booking training sessions, according to the premises?","Booking training per premise 5 requires equipment use and having a trainer. Premises 1, 3, 4, and 7 confirm Alex can use equipment, and premises 2 and 6 confirm trainer eligibility, but no premise confirms Alex has a...","[1, 3, 4, 5, 7]"
589,clean,301,1,NO,UNKNOWN,medium,phrase_unknown,True,"Based on the above premises, is the statement true?\nStatement: Exists(x, ParticipatesInStudyGroup(x))","The statement is not necessarily true because, while not every student participates in a study group, no premise confirms that at least one student does participate.",[3]
611,clean,312,1,NO,UNKNOWN,medium,phrase_unknown,True,Did Dio pass by reading books?,"Premise 5 states Dio borrowed a book and passed, premise 1 confirms he got a book, but premise 2 requires hard work to read, which is uncertain, so passing via reading isn’t guaranteed.","[1, 2, 5]"
678,clean,346,0,A,C,high,pattern:making\s+([ABCD])\s+(?:the\s+)?(?:correct|valid|strongest|best),True,"Which statement can be inferred?\nA. It is both true and false that if there exists at least one student who has completed the Thesis Writing course, then all students are enrolled in Research Methods.\nB. It is fals...","Premise 6 states all students completed Thesis Writing, implying at least one did, and premise 2 confirms all students are enrolled in Research Methods, making C correct.","[2, 6]"
680,clean,347,0,A,B,high,pattern:making\s+([ABCD])\s+(?:the\s+)?(?:correct|valid|strongest|best),True,"Which statement can be inferred?\nA. If a student doesn't complete the Thesis Writing course, then all students are enrolled in Research Methods.\nB. If a student completes the Thesis Writing course, then all student...","Premise 7 states some student completed Thesis Writing, premise 5 says this implies all students are enrolled in Research Methods, and premise 3 confirms all are enrolled, making B correct.","[3, 5, 7]"
682,clean,348,0,A,C,high,pattern:making\s+([ABCD])\s+(?:the\s+)?(?:correct|valid|strongest|best),True,"Which statement can be inferred?\nA. It is false that if all students have access to online learning resources, then they are all enrolled in at least one online course.\nB. If all students have not access to online ...","Premise 3 states all students have access to online resources, and premise 6 says this implies all are enrolled in at least one online course, making C correct.","[3, 6]"


Saved: /kaggle/working/audit_raw_vs_v5/audit_raw_explanation_label_flags.csv
Saved: /kaggle/working/audit_raw_vs_v5/audit_clean_explanation_label_flags.csv


In [6]:
# ==================================================================
# STAGE 5 -- Index validity, FOL format, and record-level structural issues
# ==================================================================
def check_idx(rec, ri, dataset):
    n = len(rec.get("premises-NL", []) or [])
    issues=[]
    for qi, idx in enumerate(rec.get("idx", []) or []):
        if idx is None:
            issues.append({"dataset":dataset,"record_i":ri,"q_idx":qi,"issue":"idx_missing","idx":idx,"n_premises":n})
            continue
        if not isinstance(idx, list):
            issues.append({"dataset":dataset,"record_i":ri,"q_idx":qi,"issue":"idx_not_list","idx":idx,"n_premises":n})
            continue
        if len(idx)==0:
            issues.append({"dataset":dataset,"record_i":ri,"q_idx":qi,"issue":"idx_empty","idx":idx,"n_premises":n})
        for k in idx:
            if not isinstance(k, int):
                issues.append({"dataset":dataset,"record_i":ri,"q_idx":qi,"issue":"idx_non_int","idx":idx,"n_premises":n})
            elif k < 1 or k > n:
                issues.append({"dataset":dataset,"record_i":ri,"q_idx":qi,"issue":"idx_out_of_range","idx":idx,"bad_idx":k,"n_premises":n})
    return issues


def fol_style_stats(data, name):
    rows=[]
    for ri, rec in enumerate(data):
        fols = rec.get("premises-FOL", []) or []
        txt = " ".join(map(str, fols))
        rows.append({
            "dataset": name,
            "record_i": ri,
            "n_fol": len(fols),
            "uses_unicode_quantifier": int("∀" in txt or "∃" in txt),
            "uses_ForAll_Exists": int("ForAll" in txt or "Exists" in txt),
            "has_arithmetic": int(bool(re.search(r"[<>]=?|=\s*\d|\d", txt))),
            "has_negation": int("¬" in txt or "Not" in txt),
        })
    return pd.DataFrame(rows)

issues=[]
for i, rec in enumerate(raw):
    issues.extend(check_idx(rec, i, "raw"))
for i, rec in enumerate(clean):
    issues.extend(check_idx(rec, i, "clean"))
struct = pd.DataFrame(issues)
folstat = pd.concat([fol_style_stats(raw, "raw"), fol_style_stats(clean, "clean")], ignore_index=True)

print("="*80)
print("IDX / FOL STRUCTURAL AUDIT")
print("="*80)
print("Structural issues total:", len(struct))
if len(struct):
    display(struct.head(50))
    display(struct.groupby(["dataset","issue"]).size().reset_index(name="count"))
else:
    print("No structural idx/FOL count issues found.")

print("FOL style stats summary:")
display(folstat.groupby("dataset")[["uses_unicode_quantifier","uses_ForAll_Exists","has_arithmetic","has_negation"]].sum())

struct.to_csv(OUT_DIR / "audit_idx_fol_structural_issues.csv", index=False)
folstat.to_csv(OUT_DIR / "audit_fol_style_stats.csv", index=False)
print("Saved:", OUT_DIR / "audit_idx_fol_structural_issues.csv")
print("Saved:", OUT_DIR / "audit_fol_style_stats.csv")


IDX / FOL STRUCTURAL AUDIT
Structural issues total: 22


,dataset,record_i,q_idx,issue,idx,n_premises
0,raw,48,1,idx_empty,[],12
1,raw,54,1,idx_empty,[],12
2,raw,57,1,idx_empty,[],12
3,raw,72,1,idx_empty,[],8
4,raw,79,0,idx_empty,[],6
5,raw,81,0,idx_empty,[],5
6,raw,81,1,idx_empty,[],5
7,raw,89,1,idx_empty,[],5
8,raw,125,1,idx_empty,[],15
9,raw,129,1,idx_empty,[],13


,dataset,issue,count
0,clean,idx_empty,11
1,raw,idx_empty,11


FOL style stats summary:


,uses_unicode_quantifier,uses_ForAll_Exists,has_arithmetic,has_negation
dataset,,,,
clean,398,0,51,304
raw,260,222,54,313


Saved: /kaggle/working/audit_raw_vs_v5/audit_idx_fol_structural_issues.csv
Saved: /kaggle/working/audit_raw_vs_v5/audit_fol_style_stats.csv


In [7]:
# ==================================================================
# STAGE 6 -- Duplicate audit
# ==================================================================
def record_signature(rec, include_answers=False):
    prem = "||".join(norm_text(x) for x in rec.get("premises-NL", []))
    qs = "||".join(norm_text(x) for x in rec.get("questions", []))
    ans = "||".join(norm_answer(x) for x in rec.get("answers", [])) if include_answers else ""
    return h(prem + "###" + qs + "###" + ans, 24)


def duplicate_groups(dataset, name):
    buckets=defaultdict(list)
    for i, rec in enumerate(dataset):
        buckets[record_signature(rec, include_answers=False)].append(i)
    rows=[]
    for sig, inds in buckets.items():
        if len(inds)>1:
            for i in inds:
                rec = dataset[i]
                rows.append({
                    "dataset": name,
                    "signature": sig,
                    "group_size": len(inds),
                    "record_i": i,
                    "answers": rec.get("answers"),
                    "q0": (rec.get("questions") or [""])[0][:200],
                    "premise0": (rec.get("premises-NL") or [""])[0][:200],
                })
    return pd.DataFrame(rows)

dup_raw = duplicate_groups(raw, "raw")
dup_clean = duplicate_groups(clean, "clean")
dups = pd.concat([dup_raw, dup_clean], ignore_index=True)

print("="*80)
print("DUPLICATE RECORD GROUPS")
print("="*80)
print("Raw duplicate rows   :", len(dup_raw))
print("Clean duplicate rows :", len(dup_clean))
if len(dups):
    display(dups.head(80))
else:
    print("No exact record-level duplicate groups found by this signature.")

dups.to_csv(OUT_DIR / "audit_duplicate_groups.csv", index=False)
print("Saved:", OUT_DIR / "audit_duplicate_groups.csv")


DUPLICATE RECORD GROUPS
Raw duplicate rows   : 10
Clean duplicate rows : 4


,dataset,signature,group_size,record_i,answers,q0,premise0
0,raw,aa9877ced456f4214b2272d3,2,8,"[B, Yes]","Based on the premises, what can we conclude about the curriculum?\nA. It enhances student engagement but not critical thinking\nB. It enhances critical thinking\nC. It needs more resources to enhance cri","If a curriculum is well-structured and has exercises, it enhances student engagement."
1,raw,aa9877ced456f4214b2272d3,2,9,"[B, Yes]","Based on the premises, what can we conclude about the curriculum?\nA. It enhances student engagement but not critical thinking\nB. It enhances critical thinking\nC. It needs more resources to enhance cri","If a curriculum is well-structured and has exercises, it enhances student engagement."
2,raw,317dd6ba551d697731982859,2,24,"[A, B]","Based on the learning science principles, which statement is correct?\nA. Spaced repetition improves both memory and academic performance\nB. Excessively long intervals preserve knowledge without review","Ebbinghaus' forgetting curve formula: R = e^(-t/S), where R is retention rate, t is elapsed time, and S is review interval."
3,raw,317dd6ba551d697731982859,2,25,"[A, B]","Based on the learning science principles, which statement is correct?\nA. Spaced repetition improves both memory and academic performance\nB. Excessively long intervals preserve knowledge without review","Ebbinghaus' forgetting curve formula: R = e^(-t/S), where R is retention rate, t is elapsed time, and S is review interval."
4,raw,76f9bb5d3ca5d8404b5b8454,2,160,"[No, No]",Do all students understand the material?,Every student is stressed during exam season.
5,raw,76f9bb5d3ca5d8404b5b8454,2,161,"[No, No]",Do all students understand the material?,Every student is stressed during exam season.
6,raw,f81db23611aa916ac3fdbc76,2,182,"[A, No]","Based on the above premises, which statement can be inferred?\nA. If all students are enrolled in the PE course, then if a person has not attended the PE course, they are not attending the PE course th",All students are enrolled in the PE course.
7,raw,f81db23611aa916ac3fdbc76,2,183,"[A, Yes]","Based on the above premises, which statement can be inferred?\nA. If all students are enrolled in the PE course, then if a person has not attended the PE course, they are not attending the PE course th",All students are enrolled in the PE course.
8,raw,c232e6d8ab54d73d1ff5e79e,2,394,"[No, A]",Do all students receive a recommendation?,"If a student gives a seminar, then they receive a recommendation."
9,raw,c232e6d8ab54d73d1ff5e79e,2,395,"[No, A]",Do all students receive a recommendation?,"If a student gives a seminar, then they receive a recommendation."


Saved: /kaggle/working/audit_raw_vs_v5/audit_duplicate_groups.csv


In [8]:
# ==================================================================
# STAGE 7 -- High-confidence repair candidates from triangle agreement
# label ↔ explanation ↔ clean changes
# ==================================================================
# Join changed labels with raw/clean explanation flags.
raw_flag_small = raw_flags[["record_i","q_idx","exp_implied","confidence","why","flag_disagree"]].rename(columns={
    "record_i":"raw_record_i", "q_idx":"raw_q_idx", "exp_implied":"raw_exp_implied", "confidence":"raw_exp_conf", "why":"raw_exp_why", "flag_disagree":"raw_exp_disagree"
})
clean_flag_small = clean_flags[["record_i","q_idx","exp_implied","confidence","why","flag_disagree"]].rename(columns={
    "record_i":"clean_record_i", "q_idx":"clean_q_idx", "exp_implied":"clean_exp_implied", "confidence":"clean_exp_conf", "why":"clean_exp_why", "flag_disagree":"clean_exp_disagree"
})

tri = matched.merge(raw_flag_small, on=["raw_record_i","raw_q_idx"], how="left")
tri = tri.merge(clean_flag_small, on=["clean_record_i","clean_q_idx"], how="left")


def tri_reason(row):
    reasons=[]
    if row.get("answer_changed") is True:
        reasons.append("raw_clean_label_changed")
    if bool(row.get("raw_exp_disagree")):
        reasons.append("raw_label_exp_conflict")
    if bool(row.get("clean_exp_disagree")):
        reasons.append("clean_label_exp_conflict")
    if pd.notna(row.get("raw_exp_implied")) and row.get("raw_exp_implied") == row.get("clean_answer_norm") and row.get("raw_answer_norm") != row.get("clean_answer_norm"):
        reasons.append("raw_exp_supports_clean_label")
    return "; ".join(reasons)

tri["review_reason"] = tri.apply(tri_reason, axis=1)
tri_review = tri[tri["review_reason"].astype(str).str.len() > 0].copy()
tri_review = tri_review.sort_values(["answer_changed","raw_exp_disagree","clean_exp_disagree"], ascending=False)

cols = [
    "raw_record_i","raw_q_idx","clean_record_i","clean_q_idx","raw_answer_norm","clean_answer_norm",
    "raw_exp_implied","raw_exp_conf","clean_exp_implied","clean_exp_conf","review_reason",
    "raw_question","raw_explanation","clean_explanation","raw_idx","clean_idx"
]
cols = [c for c in cols if c in tri_review.columns]

print("="*80)
print("TRIANGLE REVIEW CANDIDATES")
print("="*80)
print("Review candidates:", len(tri_review))
display(tri_review[cols].head(80))

tri_review[cols].to_csv(OUT_DIR / "audit_triangle_review_candidates.csv", index=False)
print("Saved:", OUT_DIR / "audit_triangle_review_candidates.csv")


TRIANGLE REVIEW CANDIDATES
Review candidates: 186


,raw_record_i,raw_q_idx,clean_record_i,clean_q_idx,raw_answer_norm,clean_answer_norm,raw_exp_implied,raw_exp_conf,clean_exp_implied,clean_exp_conf,review_reason,raw_question,raw_explanation,clean_explanation,raw_idx,clean_idx
23,306,1,299.0,1.0,NO,YES,YES,low,YES,low,raw_clean_label_changed; raw_label_exp_conflict; raw_exp_supports_clean_label,"Based on the above premises, is the statement true?\nStatement: (PracticeExercises(x) → PerformWell(x))","The statement is true because it is explicitly stated that if a student practices exercises, they will perform well in exams.","The statement is true because it is explicitly stated that if a student practices exercises, they will perform well in exams.",[7],[7]
34,250,1,248.0,1.0,NO,YES,YES,low,YES,low,raw_clean_label_changed; raw_label_exp_conflict; raw_exp_supports_clean_label,"Is the following statement true? Statement: If a student has not maintained a good academic standing, then they will not be shortlisted for an interview","If a student has not maintained good standing, they will not be shortlisted (premise 2), and at least one student has good standing (premise 3). Thus, the statement is true, requiring steps to confirm consistency.","If a student has not maintained good standing, they will not be shortlisted (premise 2), and at least one student has good standing (premise 3). Thus, the statement is true, requiring steps to confirm consistency.","[2, 3]","[2, 3]"
39,265,1,263.0,1.0,NO,YES,YES,low,YES,low,raw_clean_label_changed; raw_label_exp_conflict; raw_exp_supports_clean_label,"Based on the above premises, is the statement true?\nStatement: ((QualifiesForScholarship(x) → AttendsTraining(x)) → ForAll(x, StudiesRegularly(x)))","The statement is true because all students study regularly, making the consequent of the implication true, so the entire implication holds.","The statement is true because all students study regularly, making the consequent of the implication true, so the entire implication holds.",[2],[2]
41,243,1,241.0,1.0,NO,YES,YES,low,YES,low,raw_clean_label_changed; raw_label_exp_conflict; raw_exp_supports_clean_label,Is the following statement true? Statement: All students follow the classroom rules,"All students follow the classroom rules (premise 1), and there exists a student in the student council who follows the rules (premise 2). Thus, the statement is true, requiring steps to confirm consistency.","All students follow the classroom rules (premise 1), and there exists a student in the student council who follows the rules (premise 2). Thus, the statement is true, requiring steps to confirm consistency.","[1, 2]","[1, 2]"
44,207,1,205.0,1.0,NO,YES,YES,low,YES,low,raw_clean_label_changed; raw_label_exp_conflict; raw_exp_supports_clean_label,"Based on the above premises, is the following statement true?\nStatement: If a student publishes papers, then they conduct research.","The statement is true because it is explicitly given as a premise that if a student publishes papers, then they conduct research (premise 11).","The statement is true because it is explicitly given as a premise that if a student publishes papers, then they conduct research (premise 11).",[11],[11]
142,259,1,257.0,1.0,NO,YES,YES,low,YES,low,raw_clean_label_changed; raw_label_exp_conflict; raw_exp_supports_clean_label,"Is the following statement true? Statement: If a student has received support from the international office, then they have gained international experience","If a student received support from the international office, they gained experience (premise 3), and the existence of a student who applied (premise 5) is consistent. The statement is true, requiring steps to confirm.","If a student received support from the international office, they gained experience (premise 3), and the existence of a student who applied (premise 5) is consistent. The statement is true, requiring steps to confirm.","[3, 5]","[3, 5]"
202,275,1,273.0,1.0,NO,YES,YES,low,YES,low,raw_clean_label_chang

Saved: /kaggle/working/audit_raw_vs_v5/audit_triangle_review_candidates.csv


In [9]:
# ==================================================================
# STAGE 8 -- Z3 verifier candidate list, NOT Z3 judging
# ==================================================================
# We only list cases that are structurally safer for future Z3 verification:
# - Clean dataset
# - premises-FOL exists
# - idx valid/non-empty
# - answer is Yes/No/Unknown or MC
# - avoid obvious meta questions like "fewest premises" / "strongest" where entailment alone may be insufficient.

META_PATTERNS = [
    "fewest premises", "strongest conclusion", "strongest", "best answer", "most appropriate", "current eligibility status"
]

z3_rows=[]
for _, r in clean_q.iterrows():
    qnorm = norm_text(r.question)
    is_meta = any(p in qnorm for p in META_PATTERNS)
    idx = r.idx
    idx_ok = isinstance(idx, list) and len(idx) > 0 and all(isinstance(k, int) and 1 <= k <= r.n_premises for k in idx)
    fol_ok = r.n_fol > 0
    z3_rows.append({
        "record_i": int(r.record_i),
        "q_idx": int(r.q_idx),
        "answer": r.answer_norm,
        "idx_ok": bool(idx_ok),
        "fol_ok": bool(fol_ok),
        "is_meta_question": bool(is_meta),
        "z3_verifier_priority": "high" if (idx_ok and fol_ok and not is_meta) else ("medium" if idx_ok and fol_ok else "low"),
        "question": r.question,
        "idx": r.idx,
        "explanation": r.explanation,
    })

z3cand = pd.DataFrame(z3_rows)
print("="*80)
print("Z3 VERIFIER CANDIDATE LIST -- NOT TRUTH ORACLE")
print("="*80)
display(z3cand.groupby(["z3_verifier_priority","answer"]).size().reset_index(name="count"))
print("High-priority examples:")
display(z3cand.query("z3_verifier_priority == 'high'").head(30))

z3cand.to_csv(OUT_DIR / "audit_z3_candidate_list_clean.csv", index=False)
print("Saved:", OUT_DIR / "audit_z3_candidate_list_clean.csv")


Z3 VERIFIER CANDIDATE LIST -- NOT TRUTH ORACLE


,z3_verifier_priority,answer,count
0,high,A,107
1,high,B,46
2,high,C,20
3,high,D,16
4,high,NO,228
5,high,UNKNOWN,184
6,high,YES,169
7,low,A,2
8,low,NO,5
9,low,UNKNOWN,3


High-priority examples:


,record_i,q_idx,answer,idx_ok,fol_ok,is_meta_question,z3_verifier_priority,question,idx,explanation
1,0,1,YES,True,True,False,high,"Does it follow that if all Python projects are well-structured, then all Python projects are optimized, according to the premises?","[7, 10]","Premise 10 confirms all Python projects are well-structured. Premise 7 states that well-structured projects are optimized, implying all projects are optimized, so the statement that well-structured projects imply opt..."
3,1,1,YES,True,True,False,high,"Does Sophia qualify for the university scholarship, according to the premises?","[1, 2, 4, 5, 7, 8, 9, 10, 11]","The university scholarship per premise 5 requires an honors diploma and community service. Premise 4 requires international program eligibility and a capstone project, premise 2 requires advanced course qualification..."
4,2,0,A,True,True,False,high,"Based on the above premises, which conclusion is correct?\nA. Sophia qualifies for the university scholarship\nB. Sophia needs a faculty recommendation to qualify for the scholarship\nC. Sophia is eligible for the in...","[1, 2, 4, 5, 7, 8, 9, 10, 11]","Premises 7 and 8 confirm Sophia completed the core curriculum and passed the science assessment, satisfying premise 1 for advanced courses. Premise 9’s research methodology and premise 2 make her eligible for the int..."
5,2,1,YES,True,True,False,high,"Does Sophia qualify for the university scholarship, according to the premises?","[1, 2, 4, 5, 7, 8, 9, 10, 11]","The university scholarship per premise 5 requires an honors diploma and community service. Premise 4 requires international program eligibility and a capstone project, premise 2 requires advanced course qualification..."
6,3,0,A,True,True,False,high,"Based on the above premises, which conclusion logically follows?\nA. John qualifies for the graduate fellowship program\nB. John needs faculty recommendation for the fellowship\nC. John must complete an internship to...","[1, 2, 3, 4, 5, 6, 7]",Premise 5 and premise 1 confirm John is eligible for graduation by completing required courses. Premise 6’s GPA of 3.8 and premise 2 qualify him for honors. Premise 7’s thesis and premise 3 grant academic distinction...
7,3,1,YES,True,True,False,high,"Does John receive academic distinction, according to the premises?","[1, 2, 3, 5, 6, 7]","Academic distinction per premise 3 requires graduating with honors and completing a thesis. Premise 2 requires graduation eligibility and a GPA above 3.5, and premise 1 requires course completion. Premises 5–7 confir..."
8,4,0,C,True,True,False,high,"Based on the premises, what is the correct conclusion about Professor John?\nA. He can teach undergraduate courses but cannot supervise graduates\nB. He can serve on curriculum committees but cannot propose courses\n...","[1, 2, 3, 4, 5, 6, 7, 8]",Premise 5 and premise 1 confirm Professor John can teach undergraduate courses due to pedagogical training. Premise 6’s PhD and premise 2 allow him to supervise graduate students. Premise 7’s three or more publicatio...
9,4,1,YES,True,True,False,high,"Does Professor John meet all requirements to propose new courses, according to the premises?","[1, 2, 3, 4, 5, 6, 7, 8]","Proposing new courses per premise 4 requires serving on curriculum committees and a positive evaluation. Premise 3 requires graduate supervision and at least three publications, premise 2 requires undergraduate teach..."
10,5,0,B,True,True,False,high,"Based on the premises, what is the correct conclusion about Professor John?\nA. He can access restricted archives but cannot submit proposals\nB. He can apply for collaborative research projects\nC. He needs more pub...","[1, 2, 3, 4, 5, 6, 7, 8]","Professor John has taught for at least 5 years (Premise 5), so he is eligible for extended library access (by Premise 1). Because he has at least one publication (Premise 6), he can access restricted archives (Premis..."
11,5,1,YES,True,True,False,high,Does the logical chain 

Saved: /kaggle/working/audit_raw_vs_v5/audit_z3_candidate_list_clean.csv


In [10]:
# ==================================================================
# STAGE 9 -- Final summary and saved artifact list
# ==================================================================
summary.update({
    "matched_both": int((matched._merge == "both").sum()),
    "raw_only_removed": int((matched._merge == "left_only").sum()),
    "clean_only_added": int((matched._merge == "right_only").sum()),
    "answer_changed": int(matched.answer_changed.sum()),
    "raw_exp_label_flags": int(raw_flags.flag_disagree.sum()),
    "clean_exp_label_flags": int(clean_flags.flag_disagree.sum()),
    "structural_issues": int(len(struct)),
    "raw_duplicate_rows": int(len(dup_raw)),
    "clean_duplicate_rows": int(len(dup_clean)),
    "triangle_review_candidates": int(len(tri_review)),
    "z3_high_priority_candidates": int((z3cand.z3_verifier_priority == "high").sum()),
})

with open(OUT_DIR / "audit_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("="*80)
print("FINAL AUDIT SUMMARY")
print("="*80)
for k, v in summary.items():
    print(f"{k:32s}: {v}")

print("\nSaved artifact directory:", OUT_DIR)
for p in sorted(OUT_DIR.glob("audit_*")):
    print(" -", p.name)

print("\nNext step recommendation:")
print("1. Open audit_triangle_review_candidates.csv first.")
print("2. Manually inspect high-confidence label/explanation conflicts.")
print("3. Only after that, run Z3 verifier on audit_z3_candidate_list_clean.csv high-priority cases.")
print("4. Do NOT use Z3 alone to overwrite labels.")


FINAL AUDIT SUMMARY
raw_records                     : 411
clean_records                   : 399
raw_questions                   : 808
clean_questions                 : 784
question_delta                  : -24
record_delta                    : -12
matched_both                    : 802
raw_only_removed                : 18
clean_only_added                : 0
answer_changed                  : 83
raw_exp_label_flags             : 138
clean_exp_label_flags           : 83
structural_issues               : 22
raw_duplicate_rows              : 10
clean_duplicate_rows            : 4
triangle_review_candidates      : 186
z3_high_priority_candidates     : 770

Saved artifact directory: /kaggle/working/audit_raw_vs_v5
 - audit_clean_explanation_label_flags.csv
 - audit_duplicate_groups.csv
 - audit_fol_style_stats.csv
 - audit_idx_fol_structural_issues.csv
 - audit_label_change_pairs.csv
 - audit_label_changes_raw_vs_clean.csv
 - audit_label_distribution_shift.csv
 - audit_raw_clean_question_match